In [8]:
import xml.etree.ElementTree as ET
from pathlib import Path
from xml.dom import minidom

In [7]:
def string_to_xml_file(xml_string, file_name):
    """
    Converts a string into a well-formatted (indented) XML file, without unnecessary newlines.

    Parameters:
    xml_string (str): The XML content as a string.
    file_name (str): The desired filename for the XML file.
    """
    try:
        # Parse the XML string
        root = ET.ElementTree(ET.fromstring(xml_string))
        
        # Convert ElementTree to a string
        rough_string = ET.tostring(root.getroot(), encoding="utf-8")
        
        # Use minidom to pretty-print the XML
        parsed = minidom.parseString(rough_string)
        pretty_xml_as_string = parsed.toprettyxml(indent="  ")
        
        # Remove unnecessary blank lines created by toprettyxml()
        pretty_xml_as_string = "\n".join([line for line in pretty_xml_as_string.splitlines() if line.strip()])
        
        # Write the formatted XML to a file
        with open(file_name, "w", encoding="utf-8") as f:
            f.write(pretty_xml_as_string)
        
        print(f"XML file '{file_name}' created successfully with proper indentation and no extra newlines.")
    except ET.ParseError as e:
        print("Error parsing XML string:", e)

# Source

* Data Source:
    * Ministry of Oceans and Fisheries. 2023. Implementation Plan for the Deployment of Korean Eco-Friendly Ships (Greenship-K). (GK-2023-MOF)
    * [Effective Subsidy as of 2022: 15% construction subsidy](https://www.mybudget.go.kr/budgetBsnsInfo/executionResultView?in_year=2021&cndcy_no=E02200020&searchOrder=&searchState2=&debate_no=&searchVal=&searchSDate=&in_year=&searchCate=&searchType=&listSize=10&searchKind2=&searchState=&bmt_idx=1&page=1&pd_se=&searchEDate=&branch=&searchKind=)

* Implemented Input Files
    * `/input/policy/korea-2035/transportation/greenship_K_cp.xml`
    * `/input/policy/korea-2035/transportation/greenship_K_ep.xml`

# Greenship K

As of 2022, the effective support rate consists of a 15% construction subsidy and a 1.5 percentage point acquisition tax reduction. In the *Enhanced Ambition* scenario, we assume that from 2030 onward, the construction subsidy is assumed to increase to 30%, and the acquisition tax reduction to 2 percentage points. To ensure consistency with the GCAM model, the subsidy amount was derived by applying the effective support rate to the default construction cost values in the model, rather than using observed cost data. Detailed implementation steps are provided below.

In [4]:
proj_path = Path("../../")
xml_path = proj_path / "input" / "gcamdata" / "xml"
db_path = proj_path / "output"

In [5]:
xml_file_path = xml_path / "transportation_UCD_CORE.xml"
tree = ET.parse(xml_file_path)
root = tree.getroot()  # Get the root element of the XML
korea = root.find(".//region[@name='South Korea']")
trn_freight = korea.find(".//supplysector[@name='trn_freight']")
trn_shipping_intl = korea.find(".//supplysector[@name='trn_shipping_intl']")

In [6]:
xml_file_path = xml_path / "transportation_UCD_CORE.xml"
tree = ET.parse(xml_file_path)
root = tree.getroot()  # Get the root element of the XML
korea = root.find(".//region[@name='South Korea']")

# Create the new root for the reproduced XML
new_root = ET.Element("scenario")
new_world = ET.SubElement(new_root, "world")
new_korea = ET.SubElement(new_world, "region", {'name': "South Korea"})
for supplysector in korea.findall(".//supplysector"):
    supplysector_nm = supplysector.get('name')

    new_supplysector = ET.Element('supplysector', {'name': supplysector_nm})

    for subsector in supplysector.findall(".//tranSubsector"):
        subsector_nm = subsector.get('name')
        if subsector_nm not in ['Domestic Ship']:
            continue

        new_subsector = ET.Element('tranSubsector', {'name': subsector_nm})
        
        for stub_technology in subsector.findall(".//stub-technology"):
            stub_technology_nm = stub_technology.get("name")
            
            if stub_technology_nm == "Liquids":
                continue

            new_stub_technology = ET.Element('stub-technology', {'name': stub_technology_nm})

            interpolation_rule = ET.SubElement(new_stub_technology, 'interpolation-rule', {'apply-to': 'share-weight', 'from-year': '2020', 'to-year': '2100', 'from-value': '1', 'to-value': '1'})
            interpolation_function = ET.SubElement(interpolation_rule, 'interpolation-function', {'name': 'fixed'})

            for period in stub_technology.findall(".//period"):
                year = int(period.get('year'))
                if year <= 2020:
                    continue
                new_period = ET.SubElement(new_stub_technology, 'period', {'year': str(year)})

                tracking_non_energy_input = period.find(".//tracking-non-energy-input")
                input_cost_val = float(tracking_non_energy_input.find(".//input-cost").text)
                discount_val = str(-(input_cost_val * 0.165))
                new_minicam_non_energy_input = ET.SubElement(new_period, 'minicam-non-energy-input', {'name': 'Greenship-K'})
                new_input_cost = ET.SubElement(new_minicam_non_energy_input, 'input-cost')
                new_input_cost.text = discount_val

            if new_stub_technology:
                new_subsector.append(new_stub_technology)
        
        if new_subsector:
            new_supplysector.append(new_subsector)
    
    if new_supplysector:
        new_korea.append(new_supplysector)

In [ ]:
outfile_path = proj_path / "input" / "policy" / "korea-2035" / "transportation" / "greenship_K_cp.xml"

# save
xml_string = ET.tostring(new_root, encoding="unicode")
string_to_xml_file(xml_string, outfile_path)

XML file '/data/project/tae/gcam-core/input/policy/ndc/transportation/greenship_K_cp.xml' created successfully with proper indentation and no extra newlines.


In [35]:
xml_file_path = xml_path / "transportation_UCD_CORE.xml"
tree = ET.parse(xml_file_path)
root = tree.getroot()  # Get the root element of the XML
korea = root.find(".//region[@name='South Korea']")

# Create the new root for the reproduced XML
new_root = ET.Element("scenario")
new_world = ET.SubElement(new_root, "world")
new_korea = ET.SubElement(new_world, "region", {'name': "South Korea"})
for supplysector in korea.findall(".//supplysector"):
    supplysector_nm = supplysector.get('name')

    new_supplysector = ET.Element('supplysector', {'name': supplysector_nm})

    for subsector in supplysector.findall(".//tranSubsector"):
        subsector_nm = subsector.get('name')
        if subsector_nm not in ['Domestic Ship']:
            continue

        new_subsector = ET.Element('tranSubsector', {'name': subsector_nm})
        
        for stub_technology in subsector.findall(".//stub-technology"):
            stub_technology_nm = stub_technology.get("name")
            
            if stub_technology_nm == "Liquids":
                continue

            new_stub_technology = ET.Element('stub-technology', {'name': stub_technology_nm})

            interpolation_rule = ET.SubElement(new_stub_technology, 'interpolation-rule', {'apply-to': 'share-weight', 'from-year': '2020', 'to-year': '2100', 'from-value': '1', 'to-value': '1'})
            interpolation_function = ET.SubElement(interpolation_rule, 'interpolation-function', {'name': 'fixed'})

            for period in stub_technology.findall(".//period"):
                year = int(period.get('year'))
                if year <= 2020:
                    continue
                new_period = ET.SubElement(new_stub_technology, 'period', {'year': str(year)})

                tracking_non_energy_input = period.find(".//tracking-non-energy-input")
                input_cost_val = float(tracking_non_energy_input.find(".//input-cost").text)
                if year <= 2025:
                    discount_val = str(-(input_cost_val * 0.165))
                else:
                    discount_val = str(-(input_cost_val * 0.32))
                new_minicam_non_energy_input = ET.SubElement(new_period, 'minicam-non-energy-input', {'name': 'Greenship-K'})
                new_input_cost = ET.SubElement(new_minicam_non_energy_input, 'input-cost')
                new_input_cost.text = discount_val

            if new_stub_technology:
                new_subsector.append(new_stub_technology)
        
        if new_subsector:
            new_supplysector.append(new_subsector)
    
    if new_supplysector:
        new_korea.append(new_supplysector)

In [ ]:
outfile_path = proj_path / "input" / "policy" / "korea-2035" / "transportation" / "greenship_K_ep.xml"

# save
xml_string = ET.tostring(new_root, encoding="unicode")
string_to_xml_file(xml_string, outfile_path)

XML file '/data/project/tae/gcam-core/input/policy/ndc/transportation/greenship_K_ep.xml' created successfully with proper indentation and no extra newlines.
